# 🗄️ SQL for ML Interviews — Complete Guide
---
SQL is asked in almost EVERY ML interview! You need to know how to
query, aggregate, join, and analyze data directly from databases.

## 1️⃣ BASIC QUERIES
```sql
-- Select all data
SELECT * FROM loans;

-- Select specific columns
SELECT customer_id, income, loan_amount, default_status
FROM loans;

-- Filter with WHERE
SELECT * FROM loans WHERE default_status = 1;

-- Multiple conditions (AND, OR)
SELECT * FROM loans
WHERE income > 50000 AND credit_score > 700;

-- IN operator
SELECT * FROM loans WHERE education IN ('Graduate', 'PostGrad');

-- BETWEEN
SELECT * FROM loans WHERE age BETWEEN 25 AND 40;

-- LIKE (pattern matching)
SELECT * FROM customers WHERE name LIKE 'A%';  -- Starts with A

-- ORDER BY
SELECT * FROM loans ORDER BY income DESC LIMIT 10;

-- DISTINCT
SELECT DISTINCT education FROM loans;
```

## 2️⃣ AGGREGATION (GROUP BY) — Most Asked!
```sql
-- Count, Sum, Avg, Min, Max
SELECT education,
       COUNT(*) as total_customers,
       SUM(default_status) as total_defaults,
       AVG(income) as avg_income,
       MIN(credit_score) as min_credit,
       MAX(loan_amount) as max_loan
FROM loans
GROUP BY education
ORDER BY total_defaults DESC;

-- Default rate by education
SELECT education,
       COUNT(*) as total,
       ROUND(AVG(default_status) * 100, 2) as default_rate_pct
FROM loans
GROUP BY education
HAVING AVG(default_status) > 0.1  -- HAVING filters AFTER grouping
ORDER BY default_rate_pct DESC;

-- WHERE vs HAVING:
-- WHERE filters ROWS (before grouping)
-- HAVING filters GROUPS (after grouping)
```

## 3️⃣ JOINS — Very Important!
```sql
-- INNER JOIN (only matching records from both tables)
SELECT l.customer_id, l.loan_amount, c.name, c.city
FROM loans l
INNER JOIN customers c ON l.customer_id = c.customer_id;

-- LEFT JOIN (all from left + matching from right)
SELECT c.name, l.loan_amount
FROM customers c
LEFT JOIN loans l ON c.customer_id = l.customer_id;
-- Customers without loans will show NULL for loan_amount

-- RIGHT JOIN (all from right + matching from left)
-- FULL OUTER JOIN (all from both)

-- SELF JOIN (join table with itself)
SELECT a.customer_id, a.income, b.income as higher_income
FROM loans a
JOIN loans b ON a.income < b.income
WHERE a.education = b.education;

-- Multiple JOINs
SELECT l.customer_id, l.loan_amount, c.name, p.payment_date
FROM loans l
JOIN customers c ON l.customer_id = c.customer_id
JOIN payments p ON l.loan_id = p.loan_id;
```

## 4️⃣ WINDOW FUNCTIONS — Interviewers LOVE These!
```sql
-- ROW_NUMBER: Unique rank per row
SELECT customer_id, income,
       ROW_NUMBER() OVER (ORDER BY income DESC) as row_num
FROM loans;

-- RANK vs DENSE_RANK
SELECT customer_id, income,
       RANK() OVER (ORDER BY income DESC) as rank_val,       -- 1,2,2,4 (skips)
       DENSE_RANK() OVER (ORDER BY income DESC) as dense_val  -- 1,2,2,3 (no skip)
FROM loans;

-- PARTITION BY (rank within groups)
SELECT customer_id, education, income,
       RANK() OVER (PARTITION BY education ORDER BY income DESC) as income_rank
FROM loans;

-- Running Total
SELECT customer_id, loan_amount,
       SUM(loan_amount) OVER (ORDER BY customer_id) as running_total
FROM loans;

-- Moving Average (last 3 rows)
SELECT customer_id, income,
       AVG(income) OVER (ORDER BY customer_id ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)
       as moving_avg
FROM loans;

-- LAG / LEAD (previous/next row)
SELECT customer_id, income,
       LAG(income, 1) OVER (ORDER BY customer_id) as prev_income,
       LEAD(income, 1) OVER (ORDER BY customer_id) as next_income
FROM loans;
```

## 5️⃣ SUBQUERIES & CTEs
```sql
-- Subquery in WHERE
SELECT * FROM loans
WHERE income > (SELECT AVG(income) FROM loans);

-- Subquery in FROM
SELECT education, avg_income FROM (
    SELECT education, AVG(income) as avg_income
    FROM loans GROUP BY education
) sub
WHERE avg_income > 50000;

-- CTE (Common Table Expression) — Cleaner than subqueries!
WITH default_stats AS (
    SELECT education,
           AVG(default_status) as default_rate,
           COUNT(*) as total
    FROM loans
    GROUP BY education
)
SELECT * FROM default_stats
WHERE default_rate > 0.15
ORDER BY default_rate DESC;

-- Multiple CTEs
WITH high_income AS (
    SELECT * FROM loans WHERE income > 75000
),
low_credit AS (
    SELECT * FROM loans WHERE credit_score < 600
)
SELECT COUNT(*) FROM high_income
UNION ALL
SELECT COUNT(*) FROM low_credit;
```

## 6️⃣ CASE WHEN & Important Functions
```sql
-- CASE WHEN (if-else in SQL)
SELECT customer_id, credit_score,
       CASE
           WHEN credit_score >= 740 THEN 'Excellent'
           WHEN credit_score >= 670 THEN 'Good'
           WHEN credit_score >= 580 THEN 'Fair'
           ELSE 'Poor'
       END as credit_category
FROM loans;

-- COALESCE (handle NULLs)
SELECT customer_id, COALESCE(income, 0) as income
FROM loans;

-- String functions
SELECT UPPER(name), LOWER(email), LENGTH(name),
       SUBSTRING(name, 1, 3), TRIM(name)
FROM customers;

-- Date functions
SELECT YEAR(loan_date), MONTH(loan_date),
       DATEDIFF(CURRENT_DATE, loan_date) as days_since_loan
FROM loans;

-- CAST / CONVERT
SELECT CAST(income AS INT), CONVERT(VARCHAR, loan_date, 101)
FROM loans;
```

## 7️⃣ ML-Specific SQL Questions
```sql
-- Q: Find customers with income above average for their education
WITH avg_by_edu AS (
    SELECT education, AVG(income) as avg_inc
    FROM loans GROUP BY education
)
SELECT l.customer_id, l.income, a.avg_inc
FROM loans l
JOIN avg_by_edu a ON l.education = a.education
WHERE l.income > a.avg_inc;

-- Q: Find top 3 highest income customers per education
WITH ranked AS (
    SELECT *, ROW_NUMBER() OVER (
        PARTITION BY education ORDER BY income DESC
    ) as rn FROM loans
)
SELECT * FROM ranked WHERE rn <= 3;

-- Q: Calculate default rate trend by month
SELECT DATE_TRUNC('month', loan_date) as month,
       COUNT(*) as total,
       SUM(default_status) as defaults,
       ROUND(AVG(default_status) * 100, 2) as default_rate
FROM loans
GROUP BY DATE_TRUNC('month', loan_date)
ORDER BY month;
```